In [ ]:
# =========================================================================
# PORTFOLIO SIGNALS — trend / exit signals for cyclical holdings
# =========================================================================
# Companion to portfolio_overview.ipynb. This notebook does NOT predict
# tops — no one can. It applies a rules-based TREND-FOLLOWING discipline to
# the names where your strategy tag already says "trim into the peak":
#
#   STRATEGY ADOPTED — trend confirmation, not prediction
#   ----------------------------------------------------------------------
#   * Moving averages CONFIRM a trend has turned; they do not forecast it.
#   * Applied to CYCLE names only as an actionable trim trigger — these are
#     the holdings whose gains give back after the cycle crests, so a
#     confirmed downtrend is the evidence-based signal to trim.
#   * CATALYST names are shown for context but NOT signalled: they gap on
#     events, so lagging MAs are useless for them ("sell on the event").
#   * DCA names are shown CONTEXT-ONLY: they are hold-forever compounders, so
#     trend health is informational (e.g. to size a dip-buy) and is NEVER a
#     sell signal — trading them on trend just whipsaws you out of winners.
#
#   Signals per ticker:
#     - Price vs 50-day SMA   (fast trend)
#     - Price vs 200-day SMA  (the regime filter — Faber's classic rule)
#     - 50/200 cross          (golden = bullish, death = bearish)
#     - Verdict 🟢/🟡/🔴      (trend intact / watch / trim trigger)
#
# HONEST LIMITS: MAs lag by construction (you give back peak->signal), and
# they whipsaw in choppy markets. The job is to CAP downside on names that
# crash 40-60%, not to nail the top.
#
# Driver (optional):
#     portfolio_use = 'ai'      # 'ai' (default) or 'sector'
#     run_notebook('portfolio_signals.ipynb')
# =========================================================================

import os
import sys

try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    REPO_ROOT = '/content/drive/MyDrive/Stocks'
except ImportError:
    REPO_ROOT = os.path.dirname(os.path.abspath('__file__'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import time
import logging
import warnings

import numpy as np
import pandas as pd
import yfinance as yf
from IPython.display import HTML, display

logging.getLogger('yfinance').setLevel(logging.CRITICAL)
warnings.filterwarnings('ignore')

# =========================================================================
# 0. RESOLVE WHICH ALLOCATION
# =========================================================================
try:
    _sel_raw = portfolio_use  # noqa: F821  (may be injected by a driver cell)
except NameError:
    _sel_raw = os.environ.get('PORTFOLIO_USE', 'ai')
_sel = str(_sel_raw).strip().lower()
_AI_ALIASES = {'ai', 'ai_allocation', 'ai_allocations', 'wave', 'waves'}
_SEC_ALIASES = {'allocation', 'allocations', 'sector', 'satellite', 'sectors'}
if _sel in _AI_ALIASES:
    ALLOCATION_MODE = 'ai'
elif _sel in _SEC_ALIASES:
    ALLOCATION_MODE = 'sector'
else:
    warnings.warn(f"Unrecognized portfolio_use={_sel_raw!r}; defaulting to AI.")
    ALLOCATION_MODE = 'ai'

# =========================================================================
# 0b. RESOLVE CYCLE + CATALYST NAMES FROM THE SELECTED ALLOCATION
# =========================================================================
# Only the AI allocation carries a per-ticker STRATEGY taxonomy. The sector
# book has no per-ticker cycle/catalyst tags, so there is nothing to signal.
if ALLOCATION_MODE == 'ai':
    from portfolio.AI_allocations import (
        W1_SILICON_TARGETS, W2_POWER_TARGETS, W3_DCINFRA_TARGETS,
        W4_CLOUD_TARGETS, W5_SOFTWARE_TARGETS, W6_SPEC_TARGETS,
        STRATEGY, TARGET_WEIGHTS,
    )
    ALLOC_TITLE = 'Portfolio Signals — AI Waves'
    _BASKETS = {
        'W1_SILICON': W1_SILICON_TARGETS, 'W2_POWER': W2_POWER_TARGETS,
        'W3_DCINFRA': W3_DCINFRA_TARGETS, 'W4_CLOUD': W4_CLOUD_TARGETS,
        'W5_SOFTWARE': W5_SOFTWARE_TARGETS, 'W6_SPEC': W6_SPEC_TARGETS,
    }
    _WAVE_LABEL = {
        'W1_SILICON': 'W1 Silicon', 'W2_POWER': 'W2 Power',
        'W3_DCINFRA': 'W3 DC-Infra', 'W4_CLOUD': 'W4 Cloud',
        'W5_SOFTWARE': 'W5 Software', 'W6_SPEC': 'W6 Speculative',
    }
    # net weight per ticker = sub-weight * wave weight
    _net_w = {}
    _wave_of = {}
    for _k, _b in _BASKETS.items():
        _mw = TARGET_WEIGHTS.get(_k, 0)
        for _t, _sw in _b.items():
            _net_w[_t] = _net_w.get(_t, 0) + _sw * _mw
            _wave_of[_t] = _WAVE_LABEL.get(_k, _k)
    _STRATEGY = STRATEGY
else:  # sector — no per-ticker strategy, nothing to signal
    ALLOC_TITLE = 'Portfolio Signals — ETF/Satellite'
    _STRATEGY = {}
    _net_w = {}
    _wave_of = {}

# Cycle-position overlay (mirrors portfolio_overview). Used only to colour the
# context column; the verdict comes purely from the price/MA data.
_CYCLE_POS = {
    'NVDA': 'Mid', 'AMD': 'Mid', 'AVGO': 'Mid', 'MRVL': 'Mid', 'MU': 'Mid',
    'TSM': 'Mid', 'ASML': 'Mid', 'BESI.AS': 'Early',
    '000660.KS': 'Mid', '005930.KS': 'Mid',
    'GEV': 'Late', 'CEG': 'Late', 'VST': 'Late', 'CCJ': 'Mid', 'OKLO': 'Binary',
    'ETN': 'Early', 'PWR': 'Early', 'HUBB': 'Early', 'ABBN.SW': 'Early',
    'VRT': 'Mid', 'ANET': 'Mid', 'CRDO': 'Mid', 'COHR': 'Mid', 'NVT': 'Early',
    'ALAB': 'Mid', 'APH': 'Mid', 'CDNS': 'Mid',
}

# =========================================================================
# 1. PRICE HISTORY + MOVING-AVERAGE / CROSS SIGNALS
# =========================================================================
def _fetch_history(ticker, period='2y'):
    """Daily close series, resilient to transient yfinance failures."""
    for _attempt in range(2):
        try:
            h = yf.Ticker(ticker).history(period=period, auto_adjust=True)
            s = h['Close'].dropna()
            if len(s) >= 2:
                return s
        except Exception:
            time.sleep(0.5)
    return pd.Series(dtype=float)

def _signal_row(ticker):
    """Compute trend signals for one ticker.

    Returns dict with price, sma50, sma200, distances, cross state, and a
    verdict in {'up', 'watch', 'trim', 'na'}.
    """
    s = _fetch_history(ticker)
    out = {
        'price': None, 'sma50': None, 'sma200': None,
        'd50': None, 'd200': None, 'cross': 'n/a', 'verdict': 'na',
    }
    if len(s) < 50:
        return out
    price = float(s.iloc[-1])
    sma50 = float(s.tail(50).mean())
    sma200 = float(s.tail(200).mean()) if len(s) >= 200 else None
    out['price'] = price
    out['sma50'] = sma50
    out['sma200'] = sma200
    out['d50'] = (price / sma50 - 1) * 100
    if sma200:
        out['d200'] = (price / sma200 - 1) * 100
        # golden / death cross: compare today's 50/200 vs ~5 trading days ago
        if len(s) >= 205:
            prev50 = float(s.iloc[-6:-1].mean()) if len(s) >= 6 else sma50
            prev_window = s.iloc[:-5]
            prev200 = float(prev_window.tail(200).mean())
            now_above = sma50 >= sma200
            prev_above = prev50 >= prev200
            if now_above and not prev_above:
                out['cross'] = 'golden'
            elif (not now_above) and prev_above:
                out['cross'] = 'death'
            else:
                out['cross'] = 'above' if now_above else 'below'
        else:
            out['cross'] = 'above' if sma50 >= sma200 else 'below'

    # --- VERDICT (regime filter first, then fast trend) ---
    # 🔴 trim   : below the 200d regime line OR a fresh death cross
    # 🟡 watch  : above 200d but below the 50d (fast trend rolling over)
    # 🟢 up     : above both
    if sma200 is not None:
        if price < sma200 or out['cross'] == 'death':
            out['verdict'] = 'trim'
        elif price < sma50:
            out['verdict'] = 'watch'
        else:
            out['verdict'] = 'up'
    else:
        # not enough history for the 200d regime line — fall back to 50d only
        out['verdict'] = 'up' if price >= sma50 else 'watch'
    return out

# =========================================================================
# 2. RENDER
# =========================================================================
_VERDICT = {
    'up':    ('🟢 Trend intact', '#E6F4EA'),
    'watch': ('🟡 Below 50d — watch', '#FFF8DC'),
    'trim':  ('🔴 Below 200d / death cross — TRIM', '#FFEBEE'),
    'na':    ('— no data', '#F5F5F5'),
}
_POS_COLOR = {'Early': '#E6F4EA', 'Mid': '#FFF8DC', 'Late': '#FFE0CC',
              'Binary': '#FFEBEE'}

def _fmt_px(v):
    return f'{v:,.2f}' if v is not None else '<span style="color:#bbb;">n/a</span>'

def _fmt_d(v):
    if v is None:
        return '<span style="color:#bbb;">n/a</span>'
    color = '#1a7f37' if v >= 0 else '#c0392b'
    sign = '+' if v >= 0 else ''
    return f'<span style="color:{color};">{sign}{v:.1f}%</span>'

html = []
# Theme matched to signals.ipynb: #2C3E50 dark header, .sig-table, Arial 12px.
html.append("""<style>
.sig-table { border-collapse: collapse; width: 100%; font-family: Arial, sans-serif; font-size: 12px; }
.sig-table th { background: #2C3E50; color: white; padding: 8px 10px; text-align: left; font-weight: bold; white-space: nowrap; }
.sig-table td { padding: 6px 10px; border-bottom: 1px solid #ddd; color: #1a1a1a; white-space: nowrap; }
.sig-table tr:hover { filter: brightness(0.95); }
.sig-header { font-size: 18px; font-weight: bold; color: white; background: #2C3E50; padding: 12px 16px; border-radius: 6px 6px 0 0; }
.sig-sub { font-size: 12px; color: #ccc; background: #2C3E50; padding: 0 16px 10px; border-radius: 0 0 6px 6px; margin-bottom: 14px; }
.sig-section { font-size: 14px; font-weight: bold; color: #2C3E50; margin: 18px 0 2px; }
.sig-note { font-size: 12px; color: #555; line-height: 1.6; margin: 4px 0 6px; max-width: 900px; }
.sig-legend { font-size: 11px; color: #555; margin-top: 12px; line-height: 1.8; }
</style>""")
html.append(f'<div class="sig-header">{ALLOC_TITLE} — Trend / Exit Signals</div>')

if ALLOCATION_MODE != 'ai' or not _STRATEGY:
    html.append('<div class="sig-sub">Trend / exit signals apply only to the '
                'AI allocation, which carries a per-ticker strategy taxonomy '
                '(cycle / catalyst / dca). The sector book has no per-ticker '
                'cycle tags, so there is nothing to signal here. Set '
                'portfolio_use = "ai" to use this notebook.</div>')
else:
    html.append('<div class="sig-sub"><b>Strategy: trend confirmation, not '
                'prediction.</b> Moving averages confirm a trend has turned; '
                'they do not forecast tops. Rules-based trim discipline for '
                'cycle names — 200-day = regime filter (Faber), 50-day = fast '
                'trend. Catalyst names: context only (they gap on events). '
                'DCA names: context only (hold-forever — trend health shown for '
                'awareness, never as a sell signal).</div>')

    # Held names only (net weight > 0). This drops any zero-weight wave
    # (e.g. W6 at 0%) so the tables reflect the ACTIVE book exactly.
    def _held(strat):
        return sorted([t for t in _STRATEGY
                       if _STRATEGY[t] == strat and _net_w.get(t, 0) > 0],
                      key=lambda t: -_net_w.get(t, 0))
    _cycle = _held('cycle')
    _catalyst = _held('catalyst')
    _dca = _held('dca')

    # ---- precompute signals (one fetch per ticker) ----
    _sig = {t: _signal_row(t) for t in (_cycle + _catalyst + _dca)}

    # Wave background colours (mirrors the overview palette).
    _WAVE_BG = {
        'W1 Silicon': '#E3F2FD', 'W2 Power': '#FFF8DC', 'W3 DC-Infra': '#E0F7FA',
        'W4 Cloud': '#E8EAF6', 'W5 Software': '#FFEBEE', 'W6 Speculative': '#F3E6F5',
    }

    def _table(tickers, mode):
        # mode: 'cycle' (actionable trim), 'catalyst' (event-driven context),
        #       'dca' (hold-forever context — trend health only, never a sell).
        out = ['<table class="sig-table">']
        out.append('<tr><th>Ticker</th><th>Wave</th><th>Net %</th>'
                   '<th>Cycle</th><th>Price</th><th>vs 50d</th><th>vs 200d</th>'
                   '<th>50/200</th><th>Signal</th></tr>')
        for t in tickers:
            r = _sig[t]
            bg = _WAVE_BG.get(_wave_of.get(t), '#FFFFFF')
            pos = _CYCLE_POS.get(t)
            pos_cell = (f'<b style="background:{_POS_COLOR.get(pos, "#fff")};'
                        f'padding:1px 6px;border-radius:3px;">{pos}</b>'
                        if pos else '<span style="color:#bbb;">—</span>')
            cross_map = {
                'golden': '<b style="color:#1B5E20;">golden</b>',
                'death':  '<b style="color:#B71C1C;">death</b>',
                'above':  'above',
                'below':  '<span style="color:#B71C1C;">below</span>',
                'n/a':    '<span style="color:#bbb;">n/a</span>',
            }
            if mode == 'cycle':
                vtxt, _vbg = _VERDICT[r['verdict']]
                vcolor = {'up': '#1B5E20', 'watch': '#E65100',
                          'trim': '#B71C1C', 'na': '#888'}[r['verdict']]
            elif mode == 'catalyst':
                _ctx = {'up': '🟢 above trend', 'watch': '🟡 below 50d',
                        'trim': '⚪ below 200d (event-driven)', 'na': '— no data'}
                vtxt = _ctx[r['verdict']]
                vcolor = '#6A1B9A'
            else:  # dca — context only, framed as trend HEALTH, never "trim"
                _ctx = {'up': '🟢 healthy', 'watch': '🟡 below 50d',
                        'trim': '⚪ below 200d', 'na': '— no data'}
                vtxt = _ctx[r['verdict']]
                vcolor = '#1565C0'
            yf_url = f'https://finance.yahoo.com/quote/{t}/'
            out.append(
                f'<tr style="background:{bg};">'
                f'<td><b><a href="{yf_url}" target="_blank" '
                f'style="color:#1565C0;">{t}</a></b></td>'
                f'<td>{_wave_of.get(t, "—")}</td>'
                f'<td>{_net_w.get(t, 0) * 100:.2f}%</td>'
                f'<td>{pos_cell}</td>'
                f'<td><b>{_fmt_px(r["price"])}</b></td>'
                f'<td>{_fmt_d(r["d50"])}</td>'
                f'<td>{_fmt_d(r["d200"])}</td>'
                f'<td>{cross_map.get(r["cross"], r["cross"])}</td>'
                f'<td style="color:{vcolor};font-weight:bold;">{vtxt}</td></tr>'
            )
        out.append('</table>')
        return '\n'.join(out)

    # ---- CYCLE (actionable) ----
    html.append(f'<div class="sig-section">Cycle names — trim triggers '
                f'({len(_cycle)})</div>')
    html.append('<div class="sig-note">🟢 above both MAs = trend intact, hold. '
                '🟡 below 50d = fast trend rolling over, watch closely. '
                '🔴 below 200d or a death cross = regime has turned, '
                '<b>execute the "trim into the peak" the cycle tag calls for.</b>'
                '</div>')
    html.append(_table(_cycle, mode='cycle'))

    # ---- CATALYST (context only) ----
    html.append(f'<div class="sig-section">Catalyst names — context only '
                f'({len(_catalyst)})</div>')
    html.append('<div class="sig-note">MAs lag and these gap on events, so '
                '<b>no trim signal is derived</b>. Sell these on the thesis '
                'event (e.g. NRC approval, contract win), not on a moving '
                'average. Trend shown for situational awareness only.</div>')
    html.append(_table(_catalyst, mode='catalyst'))

    # ---- DCA (context only — full-book trend health) ----
    html.append(f'<div class="sig-section">DCA names — context only '
                f'({len(_dca)})</div>')
    html.append('<div class="sig-note"><b>Hold-forever compounders — these are '
                'NOT sell signals.</b> A drop is a discount, not a warning; you '
                'keep buying these on autopilot regardless of trend. Trend '
                'health is shown for situational awareness only (e.g. to size a '
                'dip-buy), <b>never</b> as a trigger to trim. Trading DCA names '
                'on moving averages just whipsaws you out of winners.</div>')
    html.append(_table(_dca, mode='dca'))

    # ---- legend / footnote ----
    html.append('<div class="sig-legend">')
    html.append('<b>Signals:</b> ')
    html.append('<span style="color:#1B5E20;font-weight:bold;">🟢 Trend intact</span> '
                '= above both 50d &amp; 200d SMAs | ')
    html.append('<span style="color:#E65100;font-weight:bold;">🟡 Watch</span> '
                '= below 50d (fast trend rolling over) | ')
    html.append('<span style="color:#B71C1C;font-weight:bold;">🔴 TRIM</span> '
                '= below 200d (regime filter) or a death cross')
    html.append('<br><b>50/200 cross:</b> '
                '<span style="color:#1B5E20;font-weight:bold;">golden</span> '
                '(50d crossed above 200d, bullish) | '
                '<span style="color:#B71C1C;font-weight:bold;">death</span> '
                '(50d crossed below 200d, bearish)')
    html.append('<br><b>Scope:</b> cycle names get actionable trim signals; '
                'catalyst names are context-only (sell on the event, not the MA); '
                'DCA names are context-only (hold-forever — trend health for '
                'awareness, never a sell signal).')
    html.append('<br><b>Honest limits:</b> MAs lag by construction (you give back '
                'peak&rarr;signal) and whipsaw in choppy markets. The value is '
                'capping downside on names that crash 40-60%, not calling the top. '
                'Computed from 2y daily closes (yfinance) — verify against your '
                'broker before acting. A discipline, not a forecast, not advice.')
    html.append('</div>')

display(HTML('\n'.join(html)))
